# RuteStrip — Sistem Rekomendasi Jalur Pendakian Gunung di Jawa
## Content-Based Filtering berbasis SBERT dan Cosine Similarity (versi web rutestrip.ai.id)

Notebook ini menjalankan **kode dan data yang sama dengan website**: `python/processor.py`, `python/search_scoring.py`, dan dataset `dataset/` (hasil ekspor database produksi) dari repository `dasrams31/rutestrip-app`.

Link notebook ini: https://s.peka.ai.id/rutestrip-notebook

**Alur sistem**
1. **Data GPX** → statistik jalur (jarak 3D, total tanjakan, waktu Naismith, kemiringan rata-rata, bagian paling curam ±1 km, elevasi min/maks).
2. **Tingkat kesulitan** dari profil jalur (Shenandoah difficulty rating + faktor kecuraman dan ketinggian).
3. **Narasi otomatis** dari statistik → teks konten tiap jalur.
4. **Preprocessing** (regex cleaning, case folding, stopword removal selektif, tanpa stemming).
5. **SBERT** `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensi) → embedding dokumen jalur dan per kalimat aspek.
6. **Pencarian**: nama gunung/daerah yang disebut = saringan; kata kriteria (landai, mudah, tercepat, ...) = urutan berdasarkan angka GPX; selebihnya diurutkan dengan **cosine similarity**.
7. **Evaluasi** Precision@5 dan MRR.

Versi penelitian (dataset 39 jalur di paper) tersimpan di tag git `penelitian-39-jalur`.

In [1]:
#@title 1. Setup: install library & ambil kode + data dari repository
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gpxpy', 'sentence-transformers', 'scikit-learn', 'pandas'], check=True)

REPO_DIR = os.environ.get('RUTESTRIP_REPO_DIR', 'rutestrip-app')
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/dasrams31/rutestrip-app.git', REPO_DIR], check=True)

PY_DIR = os.path.join(REPO_DIR, 'python')
DATA_DIR = os.path.join(REPO_DIR, 'dataset')
GPX_DIR = os.path.join(REPO_DIR, 'webchat-api', 'gpx_db')
if PY_DIR not in sys.path:
    sys.path.insert(0, PY_DIR)

import warnings; warnings.filterwarnings('ignore')
import inspect, re, json
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', None)

import processor        # GPX processing, Naismith, difficulty, narrative, preprocessing, SBERT
import search_scoring   # ranking yang dipakai website (sama dengan RouteSearchService.php)
import evaluate_search as ev

print('Repository :', os.path.basename(os.path.abspath(REPO_DIR)))
print('Dataset    :', sorted(os.listdir(DATA_DIR)))

Repository : rutestrip-final
Dataset    : ['.gitignore', 'README.md', 'data_dictionary.csv', 'embeddings.csv', 'gpx', 'routes.csv', 'routes.json']


## 2. Preprocessing teks
Sama dengan paper: hapus URL, karakter non-alfanumerik, angka berdiri sendiri, normalisasi spasi, case folding, lalu **stopword removal selektif** (kata negasi dan kata sifat jalur seperti *landai, curam, pemula* tetap dipertahankan). Tidak memakai stemming karena SBERT peka konteks.

In [2]:
#@title Kode `preprocess_text` (python/processor.py)
print(inspect.getsource(processor.preprocess_text))
print('Stopwords      :', len(processor.STOPWORDS_ID))
print('Preserve words :', sorted(processor.PRESERVE_WORDS))
contoh = 'Ini adalah jalur yang sangat indah dan tidak terlalu curam, cocok untuk pemula. Info: https://example.com'
print('\nAsli  :', contoh)
print('Hasil :', processor.preprocess_text(contoh))

def preprocess_text(text: str, remove_stopwords: bool = True) -> str:
    """
    Preprocessing teks sesuai paper:
    1. Data Cleaning - Regex untuk hapus karakter non-ASCII, URL, whitespace berlebih
    2. Case Folding - Lowercase
    3. Stopword Removal - Selektif (pertahankan negasi dan kata sifat krusial)
    4. NO STEMMING - Karena SBERT sensitif terhadap konteks
    """
    if not text:
        return ""
    
    # 1. Data Cleaning
    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Hapus karakter non-ASCII kecuali huruf Indonesia
    text = re.sub(r'[^\w\s\-]', ' ', text)
    # Hapus angka yang berdiri sendiri (pertahankan angka dalam konteks)
    text = re.sub(r'\b\d+\b', '', text)
    # Hapus whitespace berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Case Folding
    text = text.lower()
    
    # 3. Stopword Removal (selektif)
    if remove_stopwords:
        words = text.split()
        filtered_words = []
        for word in word

## 3. Statistik GPX, Naismith, dan tingkat kesulitan
- **Jarak**: `gpx.length_3d()` (km). Elevasi dihaluskan dulu (`gpx.smooth(vertical=True)`) untuk mengurangi noise GPS.
- **Total tanjakan**: jumlah selisih elevasi positif antar titik.
- **Naismith**: `jarak/5 + tanjakan/600` (jam), sama dengan paper.
- **Kemiringan rata-rata**: `tanjakan / (jarak × 1000) × 100%`.
- **Bagian paling curam**: kemiringan terbesar pada potongan ±1 km di sepanjang profil. Rata-rata menyembunyikan bagian curam pendek, misalnya lereng pasir Semeru.
- **Tingkat kesulitan**: skor = √(tanjakan[ft] × 2 × jarak[mil]) — *Shenandoah National Park hiking difficulty rating* — dikali 1,15 bila kemiringan rata-rata ≥ 25% dan 1,15 bila puncak ≥ 3.000 m.
  Mudah < 75 ≤ Sedang < 200 ≤ Sulit < 460 ≤ Sangat Sulit; tanjakan ≥ 1.200 m minimal **Sulit**.

Rekaman GPX yang naik lalu turun lewat jalur yang sama, berupa loop, atau berisi beberapa track sudah dirapikan menjadi satu jalur dari basecamp sampai puncak (hanya membuang titik, tidak menggeser atau menambah titik GPS).

In [3]:
#@title Kode statistik, kesulitan, dan narasi (python/processor.py)
for fn in (processor.calculate_statistics, processor.steepest_section_grade, processor.difficulty_score,
           processor.classify_difficulty, processor.generate_narrative):
    print(inspect.getsource(fn))

def calculate_statistics(gpx, points):
    """Calculate hiking statistics from GPX data"""
    
    # Jika data waktu kosong, gunakan length_3d()
    has_time = any(p['time'] is not None for p in points)
    
    if has_time:
        distance_m = gpx.length_3d()
    else:
        distance_m = gpx.length_3d()
    
    distance_km = distance_m / 1000.0
    
    # Calculate elevation gain
    elevation_gain = 0
    for i in range(1, len(points)):
        ele_diff = points[i]['ele'] - points[i-1]['ele']
        if ele_diff > 0:
            elevation_gain += ele_diff
    
    # Rumus Naismith: (Jarak_KM / 5) + (Gain_Meter / 600)
    naismith_duration = (distance_km / 5) + (elevation_gain / 600)
    
    # Rumus Grade (%): (Gain / (Jarak_KM * 1000)) * 100
    if distance_km > 0:
        average_grade = (elevation_gain / (distance_km * 1000)) * 100
    else:
        average_grade = 0
    
    # Elevation range
    elevations = [p['ele'] for p in points if p['ele'] > 0]
    min_elevation = min

## 4. Dataset jalur (sama dengan website)
`dataset/routes.json` adalah hasil `php artisan dataset:export` dari database produksi: nama jalur, lokasi titik awal (OpenStreetMap Nominatim), statistik GPX, narasi, dan sumber datanya. `dataset/embeddings.csv` berisi embedding SBERT narasi yang tersimpan di database.

In [4]:
#@title Muat dataset
raw = json.load(open(os.path.join(DATA_DIR, 'routes.json'), encoding='utf-8'))
emb = pd.read_csv(os.path.join(DATA_DIR, 'embeddings.csv'), encoding='utf-8-sig')
stored = {int(row[0]): [float(v) for v in row[1:]] for row in emb.itertuples(index=False)}

routes = []
for r in raw:
    routes.append({
        'id': r['route_id'], 'name': r['name'], 'mountain': r['mountain'], 'via': r['via'],
        'province': r['province'], 'regency': r['regency'],
        'distance_km': r['distance_km'], 'elevation_gain_m': r['elevation_gain_m'],
        'naismith_duration_hour': r['naismith_duration_hour'], 'average_grade_pct': r['average_grade_pct'],
        'steepest_grade_pct': r['steepest_grade_pct'], 'max_elevation_m': r['max_elevation_m'],
        'difficulty': r['difficulty'], 'difficulty_score': r['difficulty_score'],
        'narrative_text': r['narrative_text'], 'description': r['description'] or '',
        'source': r['source'], 'sbert_embedding': stored.get(r['route_id']),
    })

df = pd.DataFrame(routes)
print(f"Jumlah jalur : {len(df)}  |  gunung/bukit: {df['mountain'].nunique()}")
print('Provinsi     :', df['province'].value_counts().to_dict())
print('Kesulitan    :', df['difficulty'].value_counts().reindex(['Mudah', 'Sedang', 'Sulit', 'Sangat Sulit']).to_dict())
print('Sumber       :', df['source'].value_counts().to_dict())
display(df[['id', 'name', 'regency', 'distance_km', 'elevation_gain_m', 'naismith_duration_hour',
            'average_grade_pct', 'steepest_grade_pct', 'difficulty']].rename(columns={
    'name': 'Nama', 'regency': 'Kabupaten', 'distance_km': 'Jarak (km)', 'elevation_gain_m': 'Tanjakan (m)',
    'naismith_duration_hour': 'Naismith (jam)', 'average_grade_pct': 'Rata-rata (%)',
    'steepest_grade_pct': 'Paling curam (%)', 'difficulty': 'Kesulitan'}).head(15))

Jumlah jalur : 143  |  gunung/bukit: 71
Provinsi     : {'Jawa Barat': 57, 'Jawa Tengah': 50, 'Jawa Timur': 35, 'Lampung': 1}
Kesulitan    : {'Mudah': 24, 'Sedang': 52, 'Sulit': 63, 'Sangat Sulit': 4}
Sumber       : {'OpenStreetMap': 107, 'RuteStrip research collection': 36}


,id,Nama,Kabupaten,Jarak (km),Tanjakan (m),Naismith (jam),Rata-rata (%),Paling curam (%),Kesulitan
0,2,Mt. Argopuro Baderan to Bremi,Situbondo,45.10,3566,14.96,7.91,38.5,Sangat Sulit
1,3,Mt. Arjuno via Tretes,Pasuruan,11.93,2539,6.62,21.29,45.0,Sulit
2,4,Mt. Ciremai via Apuy,Majalengka,6.12,1600,3.89,26.13,44.2,Sulit
3,5,Mt. Ciremai via Linggasana,Kuningan,8.73,2418,5.78,27.72,46.7,Sulit
4,6,Mt. Ciremai via Palutungan,Kuningan,9.71,1987,5.26,20.46,45.2,Sulit
5,7,Mt. Ciremai via Sadarehe,Majalengka,6.78,1734,4.25,25.59,45.9,Sulit
6,8,Mt. Lawu via Cemoro Sewu,Magetan,6.47,1313,3.48,20.29,32.6,Sulit
7,9,Mt. Lawu via Cetho,Karanganyar,9.11,1780,4.79,19.55,34.5,Sulit
8,10,Mt. Lawu via Tambak,Karanganyar,8.69,2095,5.23,24.11,38.6,Sulit
9,11,Mt. Merbabu via Selo,Boyolali,5.54,1368,3.39,24.71,34.3,Sulit


In [5]:
#@title Cek ulang: hitung statistik langsung dari file GPX (webchat-api/gpx_db)
STOP = {'gunung', 'mt', 'mount', 'via', 'jalur', 'puncak', 'to', 'bc', 'basecamp', 'dan'}
key = lambda name: ' '.join(sorted({w for w in re.findall(r'[a-z]+', name.lower())} - STOP))
gpx_files = {}
for f in os.listdir(GPX_DIR):
    if f.endswith('.gpx'):
        gpx_files.setdefault(key(f[:-4]), []).append(f)

rows = []
for r in routes:
    hits = gpx_files.get(key(r['name']), [])
    if len(hits) != 1:
        continue
    s = processor.mode_ingest(os.path.join(GPX_DIR, hits[0]))
    if s.get('success'):
        rows.append({'Nama': r['name'], 'Jarak dataset': r['distance_km'], 'Jarak GPX': s['distance_km'],
                     'Tanjakan dataset': r['elevation_gain_m'], 'Tanjakan GPX': s['elevation_gain_m'],
                     'Naismith dataset': r['naismith_duration_hour'], 'Naismith GPX': s['naismith_duration_hour']})
check = pd.DataFrame(rows)
same = (abs(check['Jarak dataset'] - check['Jarak GPX']) <= 0.05) & (abs(check['Tanjakan dataset'] - check['Tanjakan GPX']) <= 5)
print(f"Jalur dengan file GPX yang cocok namanya: {len(check)} dari {len(routes)}")
print(f"Statistik sama dengan dataset (selisih ≤ 0,05 km dan ≤ 5 m): {same.sum()} jalur")
display(check.head(10))

Jalur dengan file GPX yang cocok namanya: 137 dari 143
Statistik sama dengan dataset (selisih ≤ 0,05 km dan ≤ 5 m): 137 jalur


,Nama,Jarak dataset,Jarak GPX,Tanjakan dataset,Tanjakan GPX,Naismith dataset,Naismith GPX
0,Mt. Arjuno via Tretes,11.93,11.93,2539,2539,6.62,6.62
1,Mt. Ciremai via Apuy,6.12,6.12,1600,1600,3.89,3.89
2,Mt. Ciremai via Linggasana,8.73,8.73,2418,2418,5.78,5.78
3,Mt. Ciremai via Palutungan,9.71,9.71,1987,1987,5.26,5.26
4,Mt. Ciremai via Sadarehe,6.78,6.78,1734,1734,4.25,4.25
5,Mt. Lawu via Cemoro Sewu,6.47,6.47,1313,1313,3.48,3.48
6,Mt. Lawu via Cetho,9.11,9.11,1780,1780,4.79,4.79
7,Mt. Lawu via Tambak,8.69,8.69,2095,2095,5.23,5.23
8,Mt. Merbabu via Selo,5.54,5.54,1368,1368,3.39,3.39
9,Mt. Merbabu via Suwanting,6.02,6.02,1772,1772,4.16,4.16


## 5. Representasi dokumen untuk SBERT
- **Dokumen jalur** = `"Jalur pendakian {nama}. {narasi} {deskripsi}"` → satu embedding.
- **Aspek** = setiap kalimat narasi (jarak, waktu, kemiringan, elevasi, kesulitan, kecocokan) + kalimat nama → satu embedding per aspek, agar query pendek seperti "jalur landai" dibandingkan dengan aspek yang paling relevan.
- Query diproses dengan `preprocess_text` yang sama seperti paper.

In [6]:
#@title Load SBERT & buat embedding dokumen + aspek
model = processor.get_model()
print('Dimensi embedding:', model.get_sentence_embedding_dimension())
print(inspect.getsource(search_scoring.build_document))

doc_vecs, aspect_vecs = {}, {}
for r in routes:
    doc = search_scoring.build_document(r)
    vecs = model.encode([doc['text']] + doc['aspects'])
    doc_vecs[r['id']] = vecs[0].tolist()
    aspect_vecs[r['id']] = vecs[1:].tolist()

contoh = search_scoring.build_document(routes[0])
print('Contoh dokumen:', contoh['text'][:200], '...')
print('Jumlah aspek  :', len(contoh['aspects']))

Dimensi embedding: 384
def build_document(route: dict) -> dict:
    """Return the text to embed for a route and its aspect sentences."""
    name = clean_route_name(route.get("name"))
    narrative = (route.get("narrative_text") or "").strip()
    description = (route.get("description") or "").strip()

    text = f"Jalur pendakian {name}. {narrative}"
    if description:
        text += f" {description}"

    aspects = [f"Jalur pendakian {name}."] + split_sentences(narrative) + split_sentences(description)[:6]
    return {"text": text, "aspects": aspects}



Contoh dokumen: Jalur pendakian Gunung Argopuro Baderan to Bremi. Jalur pendakian dengan jarak panjang sekitar 45.1 kilometer. Estimasi waktu tempuh lama selama 14.96 jam menggunakan rumus Naismith. Karakteristik jal ...
Jumlah aspek  : 7


## 6. Cara website mengurutkan hasil
1. **Saringan nama gunung**: kalau query menyebut gunung ("merbabu", "arjuna" = Arjuno), hanya jalur gunung itu yang ditampilkan.
2. **Saringan daerah**: kalau query menyebut provinsi atau kabupaten ("jawa barat", "jabar", "magelang"), hanya jalur di daerah itu.
3. **Kata kriteria** menentukan urutan dengan angka GPX:
   *landai* → bagian paling curam lalu kemiringan rata-rata naik; *curam/terjal* → kebalikannya;
   *mudah/pemula/santai* → skor kesulitan naik; *sulit/berat/menantang* → turun;
   *singkat/cepat/tektok* → waktu Naismith naik; *panjang/lama/ekspedisi* → turun.
   Bentuk *ter-* dan *-nya* ikut terbaca (*tercepat, termudah, terlandai, landainya*), juga beberapa kata Inggris (*fastest, easiest*). Negasi membalik arti ("tidak terlalu curam" = landai).
4. **Relevansi semantik** (selalu dihitung, dan menjadi urutan utama bila tidak ada kata kriteria):
   `0,4 × cos(Q, dokumen) + 0,6 × max cos(Q, aspek)`, dengan `cos(Q, D) = (Q · D) / (‖Q‖ × ‖D‖)`, dicampur kecocokan nama (0,35) dan daerah (0,4) bila disebut.

In [7]:
#@title Kode deteksi kata kriteria dan pengurutan (python/search_scoring.py)
print(inspect.getsource(search_scoring.intent_base))
print(inspect.getsource(search_scoring.detect_intents))
print(inspect.getsource(search_scoring.rank_routes))

by_id = {r['id']: r for r in routes}

def search(query, top_n=5):
    q = model.encode(search_scoring.query_text(query)).tolist()
    ranked = search_scoring.rank_routes(query, q, routes, doc_vecs, aspect_vecs, limit=top_n)
    return [(by_id[row['id']], row['score']) for row in ranked]

for q in ['jalur tercepat ke merbabu', 'jalur landai di jawa barat', 'tidak terlalu curam']:
    p = search_scoring.plan(q, routes)
    print(f"{q!r}: kriteria={p['intents']} gunung={p['mountain_tokens']} "
          f"daerah={sorted(p['regions']['provinces']) + sorted(p['regions']['regencies'])} -> {len(p['routes'])} jalur tersisa")

def intent_base(word: str):
    """The intent word behind a query word, or None: the word itself, without
    "-nya" ("landainya"), and without the superlative "ter-" ("tercepat",
    "termudah"; "teringan" is ter- + ringan)."""
    forms = [word]
    if len(word) > 5 and word.endswith("nya"):
        forms.append(word[:-3])
    for form in list(forms):
        if len(form) > 5 and form.startswith("ter"):
            forms += [form[3:], "r" + form[3:]]
    return next((f for f in forms if f in ALL_INTENT_WORDS), None)

def detect_intents(query: str) -> set:
    """Intent words. A negation in the previous two words flips the intent
    ("tidak terlalu curam" -> gentle, "tidak sulit" -> easy); a mountain name after
    "gunung"/"puncak"/... is not an intent ("Gunung Anak Krakatau"). Opposite
    intents cancel each other."""
    words = re.findall(r"[a-z]+", (query or "").lower())
    intents = set()
    for i, word in enumerate(words):
        if i > 0 and word in MOUNTAIN_AMBIGUOUS and

In [8]:
#@title 7. Coba pencarian (ubah query di kolom kanan)
QUERY = "jalur tercepat ke merbabu"  #@param {type:"string"}

print(f"Query: {QUERY!r}\n")
for i, (r, s) in enumerate(search(QUERY), 1):
    print(f"{i}. {r['name']:<44} skor {s:.3f} | {r['distance_km']} km, +{r['elevation_gain_m']} m, "
          f"{r['naismith_duration_hour']} jam, rata-rata {r['average_grade_pct']}%, paling curam {r['steepest_grade_pct']}%, {r['difficulty']}")

Query: 'jalur tercepat ke merbabu'

1. Gunung Merbabu via Wonolelo                  skor 0.805 | 4.24 km, +1309 m, 3.03 jam, rata-rata 30.87%, paling curam 42.5%, Sulit
2. Gunung Merbabu via Wekas                     skor 0.730 | 4.74 km, +1274 m, 3.07 jam, rata-rata 26.86%, paling curam 38.3%, Sulit
3. Mt. Merbabu via Selo                         skor 0.654 | 5.54 km, +1368 m, 3.39 jam, rata-rata 24.71%, paling curam 34.3%, Sulit
4. Gunung Merbabu via Cuntel                    skor 0.581 | 5.33 km, +1416 m, 3.43 jam, rata-rata 26.6%, paling curam 36.3%, Sulit
5. Gunung Merbabu via Grenden                   skor 0.505 | 5.77 km, +1580 m, 3.79 jam, rata-rata 27.42%, paling curam 45.6%, Sulit


In [9]:
#@title Contoh hasil beberapa query (Top-3)
rows = []
for q in ['jalur untuk pemula', 'jalur landai', 'jalur tercepat ke merbabu', 'jalur tersulit ke slamet',
          'pendakian panjang beberapa hari', 'jalur pendek di jawa barat', 'gede pangrango', 'jalur dengan sabana luas']:
    rows.append({'Query': q, 'Top-3': ' | '.join(f"{r['name']} ({s:.2f})" for r, s in search(q, 3))})
display(pd.DataFrame(rows))

,Query,Top-3
0,jalur untuk pemula,Bukit Gegermenjangan (0.75) | Kawah Ijen via Sunrise View Trek (0.75) | Gunung Geulis Manggahang via Arjasari (0.69)
1,jalur landai,Kawah Ijen via Sunrise View Trek (0.73) | Jalur Claket via Asitaba Pacet (0.73) | Bukit Gegermenjangan (0.73)
2,jalur tercepat ke merbabu,Gunung Merbabu via Wonolelo (0.81) | Gunung Merbabu via Wekas (0.73) | Mt. Merbabu via Selo (0.65)
3,jalur tersulit ke slamet,Gunung Slamet via Kaliwadas (0.82) | Gunung Slamet via Baturaden Lama (0.75) | Mt. Slamet via Guci (0.67)
4,pendakian panjang beberapa hari,Mt. Argopuro Baderan to Bremi (0.85) | Gunung Gede Pangrango via Tele Selatan - Sukatani (0.85) | Mt. Semeru (0.83)
5,jalur pendek di jawa barat,Gunung Geulis Manggahang via Arjasari (0.81) | Gunung Patuha via Kawah Putih (0.80) | Gunung Palasari via Palintang (0.79)
6,gede pangrango,Gunung Gede Pangrango via Tele Selatan - Sukatani (0.52) | Gunung Gede Pangrango via Cibodas (0.52) | Gunung Gede via Sukatani (0.32)
7,jalur dengan sabana luas,Jalur Cisadon Sentul (0.64) | Bukit Gragal via Cembor (0.61) | Puncak Argojembangan Muria (0.59)


## 8. Evaluasi
Kalimat uji dan label relevansi dari `python/evaluate_search.py`, diurutkan dengan cara yang sama seperti website (`rank_routes`).

Label relevansi diturunkan otomatis dari statistik GPX, sehingga dapat direproduksi:
- *landai* → kemiringan rata-rata di sepertiga terendah; *curam* → sepertiga tertinggi;
- *mudah/pemula* → skor kesulitan sepertiga terendah; *sulit* → sepertiga tertinggi;
- *singkat* / *panjang* → waktu Naismith sepertiga terendah / tertinggi;
- nama gunung → nama jalur memuat nama gunung itu; daerah → jalur di provinsi itu.

Metrik: **Precision@5** (proporsi jalur relevan di 5 teratas) dan **MRR** (rata-rata 1/peringkat jalur relevan pertama).

> **Catatan metodologi:** label relevansi dan urutan kata kriteria sama-sama memakai statistik GPX, sehingga angka ini mengukur *konsistensi urutan dengan definisi kriteria*, bukan penilaian manusia.

In [10]:
#@title Precision@5 & MRR
res, n_main, _ = ev.evaluate(routes, model, log=None)
table = pd.DataFrame({
    'Query': [row[0] for row in res['ranked']],
    'P@5': [row[1] for row in res['ranked']],
    'RR': [row[2] for row in res['ranked']],
})
table['Kelompok'] = ['utama'] * n_main + ['daerah'] * (len(table) - n_main)
display(table)

summary = []
for group, part in (('utama', slice(0, n_main)), ('daerah', slice(n_main, None))):
    rows = res['ranked'][part]
    if rows:
        summary.append({'Kelompok': group, 'Query': len(rows),
                        'P@5': round(float(np.mean([r[1] for r in rows])), 3),
                        'MRR': round(float(np.mean([r[2] for r in rows])), 3)})
summary = pd.DataFrame(summary)
display(summary)

,Query,P@5,RR,Kelompok
0,jalur landai,1.0,1.000000,utama
1,jalur untuk pemula,1.0,1.000000,utama
2,pendakian santai dan mudah,1.0,1.000000,utama
3,jalur ramah pemula tidak terlalu curam,1.0,1.000000,utama
4,jalur curam dan menantang,0.6,0.333333,utama
5,jalur sulit untuk pendaki berpengalaman,1.0,1.000000,utama
6,pendakian ekstrem,1.0,1.000000,utama
7,pendakian singkat,1.0,1.000000,utama
8,jalur pendek bisa tektok,1.0,1.000000,utama
9,jalur panjang untuk ekspedisi,1.0,1.000000,utama


,Kelompok,Query,P@5,MRR
0,utama,17,0.965,0.961
1,daerah,4,1.000,1.000


In [11]:
#@title 9. Ringkasan (dihitung dari hasil di atas)
pick = lambda g, c: summary[summary['Kelompok'] == g][c].iloc[0]
print(f"Dataset          : {len(routes)} jalur, {df['mountain'].nunique()} gunung/bukit")
print(f"Kesulitan        : " + ', '.join(f"{k} {v}" for k, v in df['difficulty'].value_counts().items()))
print(f"Query uji        : {n_main} utama + {len(table) - n_main} daerah")
print(f"Precision@5      : {pick('utama', 'P@5'):.3f}")
print(f"MRR              : {pick('utama', 'MRR'):.3f}")
if (summary['Kelompok'] == 'daerah').any():
    print(f"Query daerah     : P@5 {pick('daerah', 'P@5'):.3f}, MRR {pick('daerah', 'MRR'):.3f}")

Dataset          : 143 jalur, 71 gunung/bukit
Kesulitan        : Sulit 63, Sedang 52, Mudah 24, Sangat Sulit 4
Query uji        : 17 utama + 4 daerah
Precision@5      : 0.965
MRR              : 0.961
Query daerah     : P@5 1.000, MRR 1.000
